In [ ]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel

# Given data (theta, d, t, E):
data = np.array([
    [3.19, 2.50, 2.44, 16.0],
    [2.11, 2.10, 2.93, 32.0],
    [2.86, 1.61, 3.18, 11.7],
    [3.07, 1.56, 2.71, 23.3],
    [2.95, 2.02, 2.77, 14.8],
    [2.20, 2.13, 3.04, 24.5],
    [2.95, 2.01, 3.16, 8.4 ],
    [2.62, 2.50, 3.75, 53.9],
    [0.07, 2.58, 3.72, 1.0 ],
    [4.95, 1.35, 1.35, 50.7],
    [0.48, 2.06, 1.32, 50.2],
    [4.78, 2.60, 1.83, 3.6 ],
    [2.54, 1.00, 1.00, 57.0],
    [0.57, 1.21, 3.68, 49.3],
    [4.95, 1.00, 1.00, 7.8 ],
    [2.50, 3.00, 3.30, 1.0 ],
    [0.00, 1.00, 2.20, 27.0],
    [2.75, 1.15, 1.06, 70.4],
    [2.36, 1.10, 1.00, 56.5],
    [4.97, 1.11, 3.95, 0.9],
    [1.95, 1.63, 1.00, 71.2],
    [1.39, 1.97, 2.27, 0.0],
    [5.0,  1.0,  4.0,  0.8],
    [2.70, 2.40, 1.16, 50.7],
    [2.51, 1.89, 1.08, 93.3],
    [1.79, 2.02, 4.00, 35.7],
    [2.49, 2.04, 1.00, 87.6],
    [2.74, 1.67, 1.00, 92.3],
    [2.18, 1.32, 1.33, 68.4],
    [2.76, 1.81, 1.03, 92.2],
    [2.56, 1.91, 1.09, 92.3],
    [2.93, 1.75, 1.11, 91.7],
    [2.81, 1.62, 1.22, 93.4],
    [1.22, 1.04, 2.36, 66.3],
    [2.67, 1.75, 1.41, 93.6],
    [2.40, 1.73, 1.48, 93.8],
    [3.88, 2.14, 1.18, 38.1],
    [5.00, 1.00, 2.45, 45.1],
    [2.32, 1.86, 1.43, 93.8],
    [2.58, 1.73, 1.36, 94.2],
    [2.40, 1.73, 1.75, 94.3],
    [2.82, 1.61, 1.48, 93.9]
])

# Load dataset (adjust path if needed in your Colab environment)
df = pd.read_csv("pdata.csv", header=None)
df.columns = ['t', 'V']
# Top exploitation candidates (θ, d, t):
# [2.746 1.145 1.059]
# [2.245 1.043 1.008]
# [2.356 1.103 1.002]
# High-uncertainty candidate for exploration: [4.969 1.111 3.953]

X = data[:,0:3]   # design inputs
y = data[:,3]     # efficiency

# Fit GP with RBF kernel (normalized y):
kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(length_scale=[1,1,1], length_scale_bounds=(1e-2, 1e2))
gpr = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gpr.fit(X, y)

# Generate candidate points (random sampling in bounds):
n_samples = 20000
theta_samples = np.random.uniform(0, 5, n_samples)
d_samples     = np.random.uniform(1, 3, n_samples)
t_samples     = np.random.uniform(1, 4, n_samples)
X_cand = np.vstack([theta_samples, d_samples, t_samples]).T

# GP predictions:
mean_pred, std_pred = gpr.predict(X_cand, return_std=True)

# Identify top candidates by predicted mean (exploitation):
top_indices = np.argsort(mean_pred)[-3:][::-1]  # sorted highest first
exploit_points = X_cand[top_indices]
exploit_mean = mean_pred[top_indices]
exploit_std  = std_pred[top_indices]

# Identify top candidate by uncertainty (exploration):
best_uncert_idx = np.argmax(std_pred)
explore_point = X_cand[best_uncert_idx]
explore_mean = mean_pred[best_uncert_idx]
explore_std  = std_pred[best_uncert_idx]

print("Top exploitation candidates (θ, d, t, mean, std):")
for pt, m, s in zip(exploit_points, exploit_mean, exploit_std):
    print(f"{np.round(pt,3)} -> mean={m:.2f}, std={s:.2f}")

print("\nHigh-uncertainty candidate for exploration:")
print(f"{np.round(explore_point,3)} -> mean={explore_mean:.2f}, std={explore_std:.2f}")


Top exploitation candidates (θ, d, t, mean, std):
[2.432 1.793 1.627] -> mean=94.42, std=0.62
[2.409 1.824 1.266] -> mean=94.40, std=0.26
[2.503 1.702 1.594] -> mean=94.34, std=0.39

High-uncertainty candidate for exploration:
[4.648 2.999 3.924] -> mean=50.08, std=33.36


In [ ]:
import numpy as np
import plotly.express as px

# Given data (theta, d, t, E):
data = np.array([
    [3.19, 2.50, 2.44, 16.0],
    [2.11, 2.10, 2.93, 32.0],
    [2.86, 1.61, 3.18, 11.7],
    [3.07, 1.56, 2.71, 23.3],
    [2.95, 2.02, 2.77, 14.8],
    [2.20, 2.13, 3.04, 24.5],
    [2.95, 2.01, 3.16, 8.4 ],
    [2.62, 2.50, 3.75, 53.9],
    [0.07, 2.58, 3.72, 1.0 ],
    [4.95, 1.35, 1.35, 50.7],
    [0.48, 2.06, 1.32, 50.2],
    [4.78, 2.60, 1.83, 3.6 ],
    [2.54, 1.00, 1.00, 57.0],
    [0.57, 1.21, 3.68, 49.3],
    [4.95, 1.00, 1.00, 7.8 ],
    [2.50, 3.00, 3.30, 1.0 ],
    [0.00, 1.00, 2.20, 27.0],
    [2.75, 1.15, 1.06, 70.4],
    [2.36, 1.10, 1.00, 56.5],
    [4.97, 1.11, 3.95, 0.9],
    [1.95, 1.63, 1.00, 71.2],
    [1.39, 1.97, 2.27, 0.0],
    [5.0,  1.0,  4.0,  0.8],
    [2.70, 2.40, 1.16, 50.7],
    [2.51, 1.89, 1.08, 93.3],
    [1.79, 2.02, 4.00, 35.7],
    [2.49, 2.04, 1.00, 87.6],
    [2.74, 1.67, 1.00, 92.3],
    [2.18, 1.32, 1.33, 68.4],
    [2.76, 1.81, 1.03, 92.2],
    [2.56, 1.91, 1.09, 92.3],
    [2.93, 1.75, 1.11, 91.7],
    [2.81, 1.62, 1.22, 93.4],
    [1.22, 1.04, 2.36, 66.3],
    [2.67, 1.75, 1.41, 93.6],
    [2.40, 1.73, 1.48, 93.8],
    [3.88, 2.14, 1.18, 38.1],
    [5.00, 1.00, 2.45, 45.1],
    [2.32, 1.86, 1.43, 93.8],
    [2.58, 1.73, 1.36, 94.2],
    [2.40, 1.73, 1.75, 94.3],
    [2.82, 1.61, 1.48, 93.9]
])

theta = data[:,0]
d     = data[:,1]
t     = data[:,2]
E     = data[:,3]

# 3D interactive scatter plot
fig = px.scatter_3d(
    x=theta, y=d, z=t,
    color=E,                       # color depends on efficiency
    color_continuous_scale="Viridis", # color map
    hover_data={"θ":theta, "d":d, "t":t, "E":E}, # hover info
    labels={"x":"θ (theta)", "y":"d (clearance)", "z":"t (tension)", "color":"Efficiency E"},
    title="3D Design Space with Efficiency"
)

# Make points larger for better visibility
fig.update_traces(marker=dict(size=6, line=dict(width=1, color='DarkSlateGrey')))

# Show color bar on right
fig.update_layout(coloraxis_colorbar=dict(
    title="Efficiency E",
    thickness=20,
    tickvals=np.linspace(E.min(), E.max(), 6)
))

fig.show()


In [ ]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Given data (theta, d, t, E):
data = np.array([
    [3.19, 2.50, 2.44, 16.0],
    [2.11, 2.10, 2.93, 32.0],
    [2.86, 1.61, 3.18, 11.7],
    [3.07, 1.56, 2.71, 23.3],
    [2.95, 2.02, 2.77, 14.8],
    [2.20, 2.13, 3.04, 24.5],
    [2.95, 2.01, 3.16, 8.4 ],
    [2.62, 2.50, 3.75, 53.9],
    [0.07, 2.58, 3.72, 1.0 ],
    [4.95, 1.35, 1.35, 50.7],
    [0.48, 2.06, 1.32, 50.2],
    [4.78, 2.60, 1.83, 3.6 ],
    [2.54, 1.00, 1.00, 57.0],
    [0.57, 1.21, 3.68, 49.3],
    [4.95, 1.00, 1.00, 7.8 ],
    [2.50, 3.00, 3.30, 1.0 ],
    [0.00, 1.00, 2.20, 27.0],
    [2.75, 1.15, 1.06, 70.4],
    [2.36, 1.10, 1.00, 56.5],
    [4.97, 1.11, 3.95, 0.9],
    [1.95, 1.63, 1.00, 71.2],
    [1.39, 1.97, 2.27, 0.0],
    [5.0,  1.0,  4.0,  0.8],
    [2.70, 2.40, 1.16, 50.7],
    [2.51, 1.89, 1.08, 93.3],
    [1.79, 2.02, 4.00, 35.7],
    [2.49, 2.04, 1.00, 87.6],
    [2.74, 1.67, 1.00, 92.3],
    [2.18, 1.32, 1.33, 68.4],
    [2.76, 1.81, 1.03, 92.2],
    [2.56, 1.91, 1.09, 92.3],
    [2.93, 1.75, 1.11, 91.7],
    [2.81, 1.62, 1.22, 93.4],
    [1.22, 1.04, 2.36, 66.3],
    [2.67, 1.75, 1.41, 93.6],
    [2.40, 1.73, 1.48, 93.8],
    [3.88, 2.14, 1.18, 38.1],
    [5.00, 1.00, 2.45, 45.1],
    [2.32, 1.86, 1.43, 93.8],
    [2.58, 1.73, 1.36, 94.2],
    [2.40, 1.73, 1.75, 94.3],
    [2.82, 1.61, 1.48, 93.9]
])

theta = data[:,0]
d     = data[:,1]
t     = data[:,2]
E     = data[:,3]

# Final 3 chosen points
chosen_points = np.array([
    [2.746, 1.145, 1.059, "Exploit 1"],
    [2.356, 1.103, 1.002, "Exploit 2"],
    [4.969, 1.111, 3.953, "Explore"]
])

# Base plot with all original points
fig = px.scatter_3d(
    x=theta, y=d, z=t,
    color=E,
    color_continuous_scale="Viridis",
    hover_data={"θ":theta, "d":d, "t":t, "E":E},
    labels={"x":"θ (theta)", "y":"d (clearance)", "z":"t (tension)", "color":"Efficiency E"},
    title="3D Design Space with Efficiency and Selected New Points"
)

# Add the 3 chosen points as red markers
fig.add_trace(go.Scatter3d(
    x=chosen_points[:,0],
    y=chosen_points[:,1],
    z=chosen_points[:,2],
    mode="markers+text",
    marker=dict(size=8, color="red", symbol="diamond"),
    text=chosen_points[:,3],  # labels ("Exploit 1", etc.)
    textposition="top center",
    name="Chosen Points"
))

# Adjust colorbar
fig.update_layout(coloraxis_colorbar=dict(
    title="Efficiency E",
    thickness=20,
    tickvals=np.linspace(E.min(), E.max(), 6)
))

fig.show()


In [ ]:
import numpy as np
import pandas as pd
import torch

# ---------------------------------------------------
# Simple custom neural network (manual autograd)
# ---------------------------------------------------
class NeuralNetwork:
    def __init__(self, input_size, hidden_layers, output_size):
        self.hidden_layers = hidden_layers
        self.input_size = input_size
        self.output_size = output_size

        # ---- First hidden layer (make weight tensors LEAF tensors) ----
        fan_in, fan_out = input_size, hidden_layers[0]
        xavier_std = (2.0 / (fan_in + fan_out)) ** 0.5
        W1 = torch.randn(fan_in, fan_out)
        W1.mul_(xavier_std)      # in-place scale -> stays a leaf
        W1.requires_grad_()      # now it's a leaf with requires_grad True
        self.W1 = W1

        b1 = torch.zeros(fan_out)
        b1.requires_grad_()
        self.b1 = b1

        # ---- Middle hidden layers ----
        for i in range(1, len(hidden_layers)):
            fan_in, fan_out = hidden_layers[i - 1], hidden_layers[i]
            xavier_std = (2.0 / (fan_in + fan_out)) ** 0.5
            w_temp = torch.randn(fan_in, fan_out)
            w_temp.mul_(xavier_std)
            w_temp.requires_grad_()
            b_temp = torch.zeros(fan_out)
            b_temp.requires_grad_()
            self.__setattr__(f"W{i + 1}", w_temp)
            self.__setattr__(f"b{i + 1}", b_temp)

        # ---- Output layer ----
        fan_in, fan_out = hidden_layers[-1], output_size
        xavier_std = (2.0 / (fan_in + fan_out)) ** 0.5
        w_temp = torch.randn(fan_in, fan_out)
        w_temp.mul_(xavier_std)
        w_temp.requires_grad_()
        b_temp = torch.zeros(fan_out)
        b_temp.requires_grad_()
        self.__setattr__(f"W{len(hidden_layers) + 1}", w_temp)
        self.__setattr__(f"b{len(hidden_layers) + 1}", b_temp)


    def forward(self, x):
        """Forward pass. Sigmoid activations for simplicity."""
        current_output = x
        for i in range(len(self.hidden_layers)):
            W = getattr(self, f"W{i + 1}")
            b = getattr(self, f"b{i + 1}")
            z = current_output @ W + b  # (batch x features) @ (features x next)
            current_output = torch.sigmoid(z)

        # Output layer
        W_out = getattr(self, f"W{len(self.hidden_layers) + 1}")
        b_out = getattr(self, f"b{len(self.hidden_layers) + 1}")
        z = current_output @ W_out + b_out
        output = torch.sigmoid(z)
        return output


    def backward(self, x, y, learning_rate):
        """
        Run forward, compute MSE loss, call backward(), then update
        leaf parameters in a safe way (inside torch.no_grad()).
        """
        output = self.forward(x)
        loss = torch.nn.functional.mse_loss(output, y)

        # compute gradients (populates .grad for leaf tensors)
        loss.backward()

        # Update parameters - do not perform arithmetic directly on a tensor
        # that requires grad without torch.no_grad(); instead use no_grad block.
        for i in range(len(self.hidden_layers) + 1):
            w = getattr(self, f"W{i + 1}")
            b = getattr(self, f"b{i + 1}")

            # Make sure gradients exist (they should for leaf tensors)
            if w.grad is None or b.grad is None:
                # This should not happen now, but guard just in case.
                raise RuntimeError(f"Missing gradient for W{i+1} or b{i+1}")

            with torch.no_grad():
                w -= learning_rate * w.grad
                b -= learning_rate * b.grad

            # Zero gradients safely
            w.grad.zero_()
            b.grad.zero_()

        return loss.item()


    def training_loop(self, x, y, iterations, learning_rate):
        for itr in range(iterations):
            loss = self.backward(x, y, learning_rate)
            if itr % 100 == 0:
                print(f"Iteration {itr}, Loss: {loss:.6f}")


# -------------------------------
# Main: prepare data and train
# -------------------------------
if __name__ == "__main__":
    # Your array (rows: features..., last column is target E)
    data = np.array([
    [3.19, 2.50, 2.44, 16.0],
    [2.11, 2.10, 2.93, 32.0],
    [2.86, 1.61, 3.18, 11.7],
    [3.07, 1.56, 2.71, 23.3],
    [2.95, 2.02, 2.77, 14.8],
    [2.20, 2.13, 3.04, 24.5],
    [2.95, 2.01, 3.16, 8.4 ],
    [2.62, 2.50, 3.75, 53.9],
    [0.07, 2.58, 3.72, 1.0 ],
    [4.95, 1.35, 1.35, 50.7],
    [0.48, 2.06, 1.32, 50.2],
    [4.78, 2.60, 1.83, 3.6 ],
    [2.54, 1.00, 1.00, 57.0],
    [0.57, 1.21, 3.68, 49.3],
    [4.95, 1.00, 1.00, 7.8 ],
    [2.50, 3.00, 3.30, 1.0 ],
    [0.00, 1.00, 2.20, 27.0],
    [2.75, 1.15, 1.06, 70.4],
    [2.36, 1.10, 1.00, 56.5],
    [4.97, 1.11, 3.95, 0.9],
    [1.95, 1.63, 1.00, 71.2],
    [1.39, 1.97, 2.27, 0.0],
    [5.0,  1.0,  4.0,  0.8],
    [2.70, 2.40, 1.16, 50.7],
    [2.51, 1.89, 1.08, 93.3],
    [1.79, 2.02, 4.00, 35.7],
    [2.49, 2.04, 1.00, 87.6],
    [2.74, 1.67, 1.00, 92.3],
    [2.18, 1.32, 1.33, 68.4],
    [2.76, 1.81, 1.03, 92.2],
    [2.56, 1.91, 1.09, 92.3],
    [2.93, 1.75, 1.11, 91.7],
    [2.81, 1.62, 1.22, 93.4],
    [1.22, 1.04, 2.36, 66.3],
    [2.67, 1.75, 1.41, 93.6],
    [2.40, 1.73, 1.48, 93.8],
    [3.88, 2.14, 1.18, 38.1],
    [5.00, 1.00, 2.45, 45.1],
    [2.32, 1.86, 1.43, 93.8],
    [2.58, 1.73, 1.36, 94.2],
    [2.40, 1.73, 1.75, 94.3],
    [2.82, 1.61, 1.48, 93.9]
])

    valve_designs = pd.DataFrame(data, columns=["theta", "d", "t", "E"])

    # Features and target (reshape y to (N,1) for MSE)
    X = valve_designs.drop(columns=["E"]).to_numpy()
    y = valve_designs["E"].to_numpy().reshape(-1, 1) / 100.0  # scale to 0..1

    # Convert to torch tensors
    x = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.float32)

    # Initialize and train
    input_size = x.shape[1]         # number of features (3)
    hidden_layers = [16, 8]
    output_size = 1

    nn = NeuralNetwork(input_size, hidden_layers, output_size)

    iterations = 1000
    learning_rate = 0.01
    nn.training_loop(x, y, iterations, learning_rate)

    # Print actual vs predicted
    output = nn.forward(x)
    compare = torch.cat((y, output.detach()), dim=1)
    print("\nActual (scaled) vs Predicted")
    print(compare)


Iteration 0, Loss: 0.161952
Iteration 100, Loss: 0.144509
Iteration 200, Loss: 0.133012
Iteration 300, Loss: 0.125932
Iteration 400, Loss: 0.121731
Iteration 500, Loss: 0.119277
Iteration 600, Loss: 0.117846
Iteration 700, Loss: 0.117002
Iteration 800, Loss: 0.116494
Iteration 900, Loss: 0.116176

Actual (scaled) vs Predicted
tensor([[0.1600, 0.4961],
        [0.3200, 0.4915],
        [0.1170, 0.4899],
        [0.2330, 0.4925],
        [0.1480, 0.4931],
        [0.2450, 0.4910],
        [0.0840, 0.4909],
        [0.5390, 0.4881],
        [0.0100, 0.4855],
        [0.5070, 0.5000],
        [0.5020, 0.4967],
        [0.0360, 0.5002],
        [0.5700, 0.4990],
        [0.4930, 0.4830],
        [0.0780, 0.5008],
        [0.0100, 0.4912],
        [0.2700, 0.4864],
        [0.7040, 0.4996],
        [0.5650, 0.4991],
        [0.0090, 0.4869],
        [0.7120, 0.5004],
        [0.0000, 0.4937],
        [0.0080, 0.4865],
        [0.5070, 0.5028],
        [0.9330, 0.5017],
        [0.3570, 0.485

In [ ]:
import numpy as np
import torch
import plotly.graph_objects as go
import plotly.express as px

# ------------------------------------
# 1. Prepare original data
# ------------------------------------
data = np.array([
    [3.19, 2.50, 2.44, 16.0],
    [2.11, 2.10, 2.93, 32.0],
    [2.86, 1.61, 3.18, 11.7],
    [3.07, 1.56, 2.71, 23.3],
    [2.95, 2.02, 2.77, 14.8],
    [2.20, 2.13, 3.04, 24.5],
    [2.95, 2.01, 3.16, 8.4 ],
    [2.62, 2.50, 3.75, 53.9],
    [0.07, 2.58, 3.72, 1.0 ],
    [4.95, 1.35, 1.35, 50.7],
    [0.48, 2.06, 1.32, 50.2],
    [4.78, 2.60, 1.83, 3.6 ],
    [2.54, 1.00, 1.00, 57.0],
    [0.57, 1.21, 3.68, 49.3],
    [4.95, 1.00, 1.00, 7.8 ],
    [2.50, 3.00, 3.30, 1.0 ],
    [0.00, 1.00, 2.20, 27.0],
    [2.75, 1.15, 1.06, 70.4],
    [2.36, 1.10, 1.00, 56.5],
    [4.97, 1.11, 3.95, 0.9],
    [1.95, 1.63, 1.00, 71.2],
    [1.39, 1.97, 2.27, 0.0],
    [5.0,  1.0,  4.0,  0.8],
    [2.70, 2.40, 1.16, 50.7],
    [2.51, 1.89, 1.08, 93.3],
    [1.79, 2.02, 4.00, 35.7],
    [2.49, 2.04, 1.00, 87.6],
    [2.74, 1.67, 1.00, 92.3],
    [2.18, 1.32, 1.33, 68.4],
    [2.76, 1.81, 1.03, 92.2],
    [2.56, 1.91, 1.09, 92.3],
    [2.93, 1.75, 1.11, 91.7],
    [2.81, 1.62, 1.22, 93.4],
    [1.22, 1.04, 2.36, 66.3],
    [2.67, 1.75, 1.41, 93.6],
    [2.40, 1.73, 1.48, 93.8],
    [3.88, 2.14, 1.18, 38.1],
    [5.00, 1.00, 2.45, 45.1],
    [2.32, 1.86, 1.43, 93.8],
    [2.58, 1.73, 1.36, 94.2],
    [2.40, 1.73, 1.75, 94.3],
    [2.82, 1.61, 1.48, 93.9]
])

theta = data[:, 0]
d = data[:, 1]
t = data[:, 2]
E = data[:, 3]

# ------------------------------------
# 2. Create a dense grid of new points
# ------------------------------------
theta_grid = np.linspace(theta.min(), theta.max(), 50)
d_grid = np.linspace(d.min(), d.max(), 50)
t_grid = np.linspace(t.min(), t.max(), 50)

grid = np.array(np.meshgrid(theta_grid, d_grid, t_grid)).T.reshape(-1, 3)
grid_tensor = torch.tensor(grid, dtype=torch.float32)

# ------------------------------------
# 3. Predict efficiency using trained NN
# ------------------------------------
with torch.no_grad():
    pred_scaled = nn.forward(grid_tensor)  # scaled (0–1)
    pred = pred_scaled * 100               # back to percentage

pred_np = pred.numpy().flatten()

# Find best predicted design
best_idx = np.argmax(pred_np)
best_point = grid[best_idx]
best_eff = pred_np[best_idx]

print(f"Best predicted design:")
print(f"θ = {best_point[0]:.3f}, d = {best_point[1]:.3f}, t = {best_point[2]:.3f},  Predicted E = {best_eff:.2f}%")

# ------------------------------------
# 4. Plot 3D scatter with best point
# ------------------------------------
fig = px.scatter_3d(
    x=theta, y=d, z=t, color=E,
    color_continuous_scale="Viridis",
    labels={"x":"θ (theta)", "y":"d (clearance)", "z":"t (tension)", "color":"Efficiency E"},
    title="NN-Predicted Design Space with Best Candidate"
)

# Add best predicted point
fig.add_trace(go.Scatter3d(
    x=[best_point[0]],
    y=[best_point[1]],
    z=[best_point[2]],
    mode="markers+text",
    marker=dict(size=8, color="red", symbol="diamond"),
    text=[f"Best NN: {best_eff:.1f}%"],
    textposition="top center",
    name="Best NN Prediction"
))

fig.update_layout(
    scene=dict(
        xaxis_title="θ (theta)",
        yaxis_title="d (clearance)",
        zaxis_title="t (tension)"
    ),
    coloraxis_colorbar=dict(title="Efficiency E"),
    legend=dict(x=0, y=1)
)

fig.show()


Best predicted design:
θ = 3.776, d = 3.000, t = 1.000,  Predicted E = 50.53%


In [ ]:
# week 11
"""
nn_candidate_search.py

Requirements:
- python >= 3.8
- numpy, pandas, torch, scikit-learn, plotly

Install (if needed):
pip install numpy pandas torch scikit-learn plotly
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import os

# -------------------------
# User data (replace / extend as needed)
# -------------------------
data = np.array([
    [3.19, 2.50, 2.44, 16.0],
    [2.11, 2.10, 2.93, 32.0],
    [2.86, 1.61, 3.18, 11.7],
    [3.07, 1.56, 2.71, 23.3],
    [2.95, 2.02, 2.77, 14.8],
    [2.20, 2.13, 3.04, 24.5],
    [2.95, 2.01, 3.16, 8.4 ],
    [2.62, 2.50, 3.75, 53.9],
    [0.07, 2.58, 3.72, 1.0 ],
    [4.95, 1.35, 1.35, 50.7],
    [0.48, 2.06, 1.32, 50.2],
    [4.78, 2.60, 1.83, 3.6 ],
    [2.54, 1.00, 1.00, 57.0],
    [0.57, 1.21, 3.68, 49.3],
    [4.95, 1.00, 1.00, 7.8 ],
    [2.50, 3.00, 3.30, 1.0 ],
    [0.00, 1.00, 2.20, 27.0],
    [2.75, 1.15, 1.06, 70.4],
    [2.36, 1.10, 1.00, 56.5],
    [4.97, 1.11, 3.95, 0.9],
    [1.95, 1.63, 1.00, 71.2],
    [1.39, 1.97, 2.27, 0.0],
    [5.0,  1.0,  4.0,  0.8],
    [2.70, 2.40, 1.16, 50.7],
    [2.51, 1.89, 1.08, 93.3],
    [1.79, 2.02, 4.00, 35.7],
    [2.49, 2.04, 1.00, 87.6],
    [2.74, 1.67, 1.00, 92.3],
    [2.18, 1.32, 1.33, 68.4],
    [2.76, 1.81, 1.03, 92.2],
    [2.56, 1.91, 1.09, 92.3],
    [2.93, 1.75, 1.11, 91.7],
    [2.81, 1.62, 1.22, 93.4],
    [1.22, 1.04, 2.36, 66.3],
    [2.67, 1.75, 1.41, 93.6],
    [2.40, 1.73, 1.48, 93.8],
    [3.88, 2.14, 1.18, 38.1],
    [5.00, 1.00, 2.45, 45.1],
    [2.32, 1.86, 1.43, 93.8],
    [2.58, 1.73, 1.36, 94.2],
    [2.40, 1.73, 1.75, 94.3],
    [2.82, 1.61, 1.48, 93.9]
])

df = pd.DataFrame(data, columns=["theta", "d", "t", "E"])

# -------------------------
# 1) pick best-3 by E (existing)
# -------------------------
best3_idx = df["E"].nlargest(3).index.tolist()
best3 = df.loc[best3_idx, ["theta", "d", "t"]].to_numpy()
print("Best 3 existing points (by measured E):")
print(df.loc[best3_idx, :])

# -------------------------
# 2) Generate 100 candidate points near convex hull of best3
#    Method: sample random convex weights for best3, then add small Gaussian noise
# -------------------------
def sample_near_convex(points, n_samples=100, noise_scale=0.05, bounds=None):
    """
    points: (3, dim) array (best3)
    n_samples: number to sample
    noise_scale: relative noise magnitude (fraction of bounding box length)
    bounds: optional ((min_vec),(max_vec)) to clip results
    """
    m, dim = points.shape
    assert m == 3, "Expect best3 shape (3, dim)."
    # Sample random convex weights
    weights = np.random.dirichlet(alpha=[1.0]*m, size=n_samples)  # shape (n_samples, 3)
    base = weights @ points  # convex combos
    # Add Gaussian noise scaled to data range
    data_range = points.max(axis=0) - points.min(axis=0)
    # if zero range in some dim, give small epsilon
    data_range = np.where(data_range == 0, 1e-3, data_range)
    noise = np.random.randn(n_samples, dim) * (noise_scale * data_range)
    cand = base + noise
    if bounds is not None:
        minb, maxb = np.array(bounds[0]), np.array(bounds[1])
        cand = np.clip(cand, minb, maxb)
    return cand

# Set bounds for design variables (as you specified earlier)
bounds_min = np.array([0.0, 1.0, 1.0])
bounds_max = np.array([5.0, 3.0, 4.0])

candidates = sample_near_convex(best3, n_samples=100, noise_scale=0.06, bounds=(bounds_min, bounds_max))
candidates_df = pd.DataFrame(candidates, columns=["theta", "d", "t"])
print("\nSampled candidates (first 5):")
print(candidates_df.head())

# -------------------------
# 3) Build a robust PyTorch NN model (simple MLP) and training utilities
# -------------------------
class MLP(nn.Module):
    def __init__(self, input_dim=3, hidden_sizes=[64,32], output_dim=1):
        super().__init__()
        layers = []
        last = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(last, h))
            layers.append(nn.ReLU())
            last = h
        layers.append(nn.Linear(last, output_dim))
        # no final activation (we'll predict E scaled)
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

# -------------------------
# 4) Prepare data (scaling)
# -------------------------
X = df[["theta","d","t"]].to_numpy()
y = df["E"].to_numpy().reshape(-1,1).astype(np.float32) / 100.0  # scale to 0..1

scaler_X = StandardScaler()
Xs = scaler_X.fit_transform(X)
cand_Xs = scaler_X.transform(candidates)  # scale candidates the same way

X_tensor = torch.tensor(Xs, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

# -------------------------
# 5) Training loop with Adam, scheduler and early stopping
# -------------------------
def train_model(model, X_tensor, y_tensor, lr=1e-3, weight_decay=1e-5,
                batch_size=32, epochs=2000, patience=200, verbose=True):
    model = model
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=100)
    best_loss = float("inf")
    best_state = None
    wait = 0
    N = X_tensor.shape[0]

    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(N)
        epoch_loss = 0.0
        # minibatch
        for i in range(0, N, batch_size):
            idx = perm[i:i+batch_size]
            xb = X_tensor[idx]
            yb = y_tensor[idx]
            preds = model(xb)
            loss = criterion(preds, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            epoch_loss += loss.item() * xb.shape[0]
        epoch_loss /= N
        scheduler.step(epoch_loss)

        if verbose and (epoch % 200 == 0 or epoch == epochs-1):
            print(f"Epoch {epoch:4d}  loss={epoch_loss:.6f}  lr={opt.param_groups[0]['lr']:.5g}")

        # early stopping
        if epoch_loss < best_loss - 1e-8:
            best_loss = epoch_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait > patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch}, best loss {best_loss:.6f}")
                break
    # load best
    model.load_state_dict(best_state)
    return model, best_loss

# -------------------------
# 6) Instantiate & train
# -------------------------
torch.manual_seed(0)
model = MLP(input_dim=3, hidden_sizes=[64,32], output_dim=1)
model, best_loss = train_model(model, X_tensor, y_tensor, lr=1e-3, weight_decay=1e-6,
                              batch_size=16, epochs=4000, patience=500, verbose=True)
print("Training finished. Best loss:", best_loss)

# -------------------------
# 7) Predict on candidates and rank them
# -------------------------
model.eval()
with torch.no_grad():
    cand_tensor = torch.tensor(cand_Xs, dtype=torch.float32)
    pred_cand = model(cand_tensor).numpy().reshape(-1) * 100.0  # scale back to 0..100

candidates_df["pred_E"] = pred_cand
# Add also a crude uncertainty proxy: ensemble or MC-dropout recommended,
# but here we give a simple local-variance proxy: distance to nearest train point
from sklearn.neighbors import NearestNeighbors
nnbr = NearestNeighbors(n_neighbors=1).fit(X)
dists, _ = nnbr.kneighbors(candidates_df[["theta","d","t"]].to_numpy())
candidates_df["dist_to_train"] = dists.reshape(-1)

# Rank by predicted E
candidates_df_sorted = candidates_df.sort_values("pred_E", ascending=False).reset_index(drop=True)
print("\nTop candidate predictions (by NN):")
print(candidates_df_sorted.head(10))

# Choose top 5 and final top 3
top5 = candidates_df_sorted.head(5)
final3 = candidates_df_sorted.head(3)
print("\nFinal 3 selected (NN best guesses):")
print(final3[["theta","d","t","pred_E","dist_to_train"]])

# -------------------------
# 8) Plot: original points colored by E, sampled candidates (small), top picks highlighted
# Save interactive HTML
# -------------------------
import plotly.express as px
import plotly.graph_objects as go

fig = go.Figure()

# original data
fig.add_trace(go.Scatter3d(
    x=df["theta"], y=df["d"], z=df["t"],
    mode="markers",
    marker=dict(size=5, color=df["E"], colorscale="Viridis", colorbar=dict(title="E")),
    text=[f"E={e:.1f}" for e in df["E"]],
    name="Existing"
))

# all candidates (small)
fig.add_trace(go.Scatter3d(
    x=candidates_df["theta"], y=candidates_df["d"], z=candidates_df["t"],
    mode="markers",
    marker=dict(size=3, color=candidates_df["pred_E"], colorscale="Cividis", opacity=0.7),
    text=[f"predE={pe:.1f}" for pe in candidates_df["pred_E"]],
    name="Candidates"
))

# top5
fig.add_trace(go.Scatter3d(
    x=top5["theta"], y=top5["d"], z=top5["t"],
    mode="markers+text",
    marker=dict(size=7, color="red", symbol="diamond"),
    text=[f"{i+1}" for i in range(len(top5))],
    textposition="top center",
    name="Top5"
))

fig.update_layout(scene=dict(xaxis_title="theta", yaxis_title="d", zaxis_title="t"),
                  title="Original points, Candidates, and Top NN picks")

out_html = "nn_candidates_plot.html"
fig.write_html(out_html, include_plotlyjs="cdn")
print(f"\nInteractive plot saved as: {out_html}  (open in browser)")

# save final3 to CSV for experiments
final3.to_csv("final3_candidates.csv", index=False)
print("Saved final3_candidates.csv")

Best 3 existing points (by measured E):
    theta     d     t     E
40   2.40  1.73  1.75  94.3
39   2.58  1.73  1.36  94.2
41   2.82  1.61  1.48  93.9

Sampled candidates (first 5):
      theta         d         t
0  2.595231  1.715205  1.339788
1  2.535769  1.713195  1.531191
2  2.459488  1.718678  1.722716
3  2.590074  1.705086  1.581807
4  2.761815  1.643215  1.455963
Epoch    0  loss=0.232892  lr=0.001
Epoch  200  loss=0.000406  lr=0.001
Epoch  400  loss=0.000112  lr=0.001
Epoch  600  loss=0.000074  lr=0.001
Epoch  800  loss=0.000079  lr=0.001
Epoch 1000  loss=0.000084  lr=0.001
Epoch 1200  loss=0.000088  lr=0.001
Epoch 1400  loss=0.000020  lr=0.0005
Epoch 1600  loss=0.000022  lr=0.0005
Epoch 1800  loss=0.000018  lr=0.0005
Epoch 2000  loss=0.000043  lr=0.0005
Epoch 2200  loss=0.000033  lr=0.0005
Epoch 2400  loss=0.000002  lr=0.00025
Epoch 2600  loss=0.000002  lr=0.00025
Epoch 2800  loss=0.000003  lr=0.00025
Epoch 3000  loss=0.000001  lr=0.000125
Epoch 3200  loss=0.000001  lr=0.000

In [ ]:
# week 11
# 8) Plot: original points colored by E, sampled candidates (small), top picks highlighted
# -------------------------
import plotly.graph_objects as go

fig = go.Figure()

# original data
fig.add_trace(go.Scatter3d(
    x=df["theta"], y=df["d"], z=df["t"],
    mode="markers",
    marker=dict(size=5, color=df["E"], colorscale="Viridis", colorbar=dict(title="E")),
    text=[f"E={e:.1f}" for e in df["E"]],
    name="Existing"
))

# all candidates (small)
fig.add_trace(go.Scatter3d(
    x=candidates_df["theta"], y=candidates_df["d"], z=candidates_df["t"],
    mode="markers",
    marker=dict(size=3, color=candidates_df["pred_E"], colorscale="Cividis", opacity=0.7),
    text=[f"predE={pe:.1f}" for pe in candidates_df["pred_E"]],
    name="Candidates"
))

# top5 (highlighted red diamonds)
fig.add_trace(go.Scatter3d(
    x=top5["theta"], y=top5["d"], z=top5["t"],
    mode="markers+text",
    marker=dict(size=8, color="red", symbol="diamond"),
    text=[f"{i+1}" for i in range(len(top5))],
    textposition="top center",
    name="Top5"
))

fig.update_layout(
    scene=dict(xaxis_title="θ", yaxis_title="d", zaxis_title="t"),
    title="Original Points, Candidates, and Top NN Picks",
    legend=dict(x=0.02, y=0.98)
)

fig.show()  # <--- HTML fayl o‘rniga shu


In [ ]:
# week 12
"""
nn_candidate_search.py

Requirements:
- python >= 3.8
- numpy, pandas, torch, scikit-learn, plotly

Install (if needed):
pip install numpy pandas torch scikit-learn plotly
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import os

# -------------------------
# User data (replace / extend as needed)
# -------------------------
data = np.array([
    [3.19, 2.50, 2.44, 16.0],
    [2.11, 2.10, 2.93, 32.0],
    [2.86, 1.61, 3.18, 11.7],
    [3.07, 1.56, 2.71, 23.3],
    [2.95, 2.02, 2.77, 14.8],
    [2.20, 2.13, 3.04, 24.5],
    [2.95, 2.01, 3.16, 8.4 ],
    [2.62, 2.50, 3.75, 53.9],
    [0.07, 2.58, 3.72, 1.0 ],
    [4.95, 1.35, 1.35, 50.7],
    [0.48, 2.06, 1.32, 50.2],
    [4.78, 2.60, 1.83, 3.6 ],
    [2.54, 1.00, 1.00, 57.0],
    [0.57, 1.21, 3.68, 49.3],
    [4.95, 1.00, 1.00, 7.8 ],
    [2.50, 3.00, 3.30, 1.0 ],
    [0.00, 1.00, 2.20, 27.0],
    [2.75, 1.15, 1.06, 70.4],
    [2.36, 1.10, 1.00, 56.5],
    [4.97, 1.11, 3.95, 0.9],
    [1.95, 1.63, 1.00, 71.2],
    [1.39, 1.97, 2.27, 0.0],
    [5.0,  1.0,  4.0,  0.8],
    [2.70, 2.40, 1.16, 50.7],
    [2.51, 1.89, 1.08, 93.3],
    [1.79, 2.02, 4.00, 35.7],
    [2.49, 2.04, 1.00, 87.6],
    [2.74, 1.67, 1.00, 92.3],
    [2.18, 1.32, 1.33, 68.4],
    [2.76, 1.81, 1.03, 92.2],
    [2.56, 1.91, 1.09, 92.3],
    [2.93, 1.75, 1.11, 91.7],
    [2.81, 1.62, 1.22, 93.4],
    [1.22, 1.04, 2.36, 66.3],
    [2.67, 1.75, 1.41, 93.6],
    [2.40, 1.73, 1.48, 93.8],
    [3.88, 2.14, 1.18, 38.1],
    [5.00, 1.00, 2.45, 45.1],
    [2.32, 1.86, 1.43, 93.8],
    [2.58, 1.73, 1.36, 94.2],
    [2.40, 1.73, 1.75, 94.3],
    [2.82, 1.61, 1.48, 93.9],
    [2.90, 1.73, 1.55, 94.6],
    [2.40, 1.73, 1.80, 93.7],
    [2.87, 1.58, 1.15, 92.6]
])

df = pd.DataFrame(data, columns=["theta", "d", "t", "E"])

# -------------------------
# 1) pick best-3 by E (existing)
# -------------------------
best3_idx = df["E"].nlargest(3).index.tolist()
best3 = df.loc[best3_idx, ["theta", "d", "t"]].to_numpy()
print("Best 3 existing points (by measured E):")
print(df.loc[best3_idx, :])

# -------------------------
# 1) pick best-3 by E (existing)
# -------------------------
best3_idx = df["E"].nlargest(3).index.tolist()
best3 = df.loc[best3_idx, ["theta", "d", "t"]].to_numpy()
print("Best 3 existing points (by measured E):")
print(df.loc[best3_idx, :])

# 2) Uniformy meshing 100 candidate points by tension t around the best point number #1
#    Method: aniq method o'ylamadim hozircha, faqat shu unirom 100 pointsni NN uchun
#    predict qilishga feed qilishni o'yladim as oppsed to randomly genarting 100 pts last time
# -------------------------
def sample_around_best():
    """
    points: (3, dim) array (best3)
    n_samples: number to sample
    noise_scale: relative noise magnitude (fraction of bounding box length)
    bounds: optional ((min_vec),(max_vec)) to clip results
    bu yerga ham yaxshi doc yozilsin please
    """
    # print(best3[0][2]-best3[1][2])
    perturb = best3[0][2]-best3[1][2]
    cand = np.linspace(best3[0][2]-perturb, best3[0][2]+perturb, 100) # instead of randomly generating 100, here meshing it
    return cand


Best 3 existing points (by measured E):
    theta     d     t     E
42   2.90  1.73  1.55  94.6
40   2.40  1.73  1.75  94.3
39   2.58  1.73  1.36  94.2
Best 3 existing points (by measured E):
    theta     d     t     E
42   2.90  1.73  1.55  94.6
40   2.40  1.73  1.75  94.3
39   2.58  1.73  1.36  94.2


In [ ]:
# week 12
"""
nn_candidate_search.py

Requirements:
- python >= 3.8
- numpy, pandas, torch, scikit-learn, plotly

Install (if needed):
pip install numpy pandas torch scikit-learn plotly
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import os

# -------------------------
# User data (replace / extend as needed)
# -------------------------
data = np.array([
    [3.19, 2.50, 2.44, 16.0],
    [2.11, 2.10, 2.93, 32.0],
    [2.86, 1.61, 3.18, 11.7],
    [3.07, 1.56, 2.71, 23.3],
    [2.95, 2.02, 2.77, 14.8],
    [2.20, 2.13, 3.04, 24.5],
    [2.95, 2.01, 3.16, 8.4 ],
    [2.62, 2.50, 3.75, 53.9],
    [0.07, 2.58, 3.72, 1.0 ],
    [4.95, 1.35, 1.35, 50.7],
    [0.48, 2.06, 1.32, 50.2],
    [4.78, 2.60, 1.83, 3.6 ],
    [2.54, 1.00, 1.00, 57.0],
    [0.57, 1.21, 3.68, 49.3],
    [4.95, 1.00, 1.00, 7.8 ],
    [2.50, 3.00, 3.30, 1.0 ],
    [0.00, 1.00, 2.20, 27.0],
    [2.75, 1.15, 1.06, 70.4],
    [2.36, 1.10, 1.00, 56.5],
    [4.97, 1.11, 3.95, 0.9],
    [1.95, 1.63, 1.00, 71.2],
    [1.39, 1.97, 2.27, 0.0],
    [5.0,  1.0,  4.0,  0.8],
    [2.70, 2.40, 1.16, 50.7],
    [2.51, 1.89, 1.08, 93.3],
    [1.79, 2.02, 4.00, 35.7],
    [2.49, 2.04, 1.00, 87.6],
    [2.74, 1.67, 1.00, 92.3],
    [2.18, 1.32, 1.33, 68.4],
    [2.76, 1.81, 1.03, 92.2],
    [2.56, 1.91, 1.09, 92.3],
    [2.93, 1.75, 1.11, 91.7],
    [2.81, 1.62, 1.22, 93.4],
    [1.22, 1.04, 2.36, 66.3],
    [2.67, 1.75, 1.41, 93.6],
    [2.40, 1.73, 1.48, 93.8],
    [3.88, 2.14, 1.18, 38.1],
    [5.00, 1.00, 2.45, 45.1],
    [2.32, 1.86, 1.43, 93.8],
    [2.58, 1.73, 1.36, 94.2],
    [2.40, 1.73, 1.75, 94.3],
    [2.82, 1.61, 1.48, 93.9],
    [2.90, 1.73, 1.55, 94.6],
    [2.40, 1.73, 1.80, 93.7],
    [2.87, 1.58, 1.15, 92.6]
])

df = pd.DataFrame(data, columns=["theta", "d", "t", "E"])

# -------------------------
# 1) Pick best-3 by E (existing)
# -------------------------
best3_idx = df["E"].nlargest(3).index.tolist()
best3 = df.loc[best3_idx, ["theta", "d", "t"]].to_numpy()
print("Best 3 existing points (by measured E):")
print(df.loc[best3_idx, :])

# -------------------------
# 2) Uniformly mesh 100 candidate points by tension t around the best point (#1)
# -------------------------
def sample_around_best_tension(best3, n_samples=100, bounds=None):
    """
    Generate candidate points by uniformly meshing the tension (t) variable
    around the best observed point, keeping θ (theta) and d fixed.

    Parameters
    ----------
    best3 : np.ndarray
        Array of the top-3 points with shape (3, 3): columns [theta, d, t].
        The first row (best3[0]) is used as the anchor.
    n_samples : int, optional
        Number of candidate points to generate (default 100).
    bounds : tuple((min_vec), (max_vec)), optional
        Optional lower/upper bounds for [theta, d, t]; candidates are clipped to these.

    Returns
    -------
    candidates : np.ndarray of shape (n_samples, 3)
        Candidate points generated by meshing t around the best point.
    """
    base_theta, base_d, base_t = best3[0]
    # Use symmetric perturbation range determined from the difference between best #1 and #2
    perturb = abs(best3[0, 2] - best3[1, 2])
    # Create uniform grid around t_best ± perturb
    t_values = np.linspace(base_t - perturb, base_t + perturb, n_samples)
    # Combine into candidate array
    candidates = np.column_stack([
        np.full_like(t_values, base_theta),
        np.full_like(t_values, base_d),
        t_values
    ])
    # Clip to global bounds if provided
    if bounds is not None:
        minb, maxb = bounds
        candidates = np.clip(candidates, minb, maxb)
    return candidates

# Set bounds for design variables
bounds_min = np.array([0.0, 1.0, 1.0])
bounds_max = np.array([5.0, 3.0, 4.0])

# Generate tension-based candidates
candidates = sample_around_best_tension(best3, n_samples=100, bounds=(bounds_min, bounds_max))
candidates_df = pd.DataFrame(candidates, columns=["theta", "d", "t"])
print("\nSampled candidates (first 5):")
print(candidates_df.head())

# -------------------------
# 3) Build a robust PyTorch NN model (simple MLP) and training utilities
# -------------------------
class MLP(nn.Module):
    def __init__(self, input_dim=3, hidden_sizes=[64,32], output_dim=1):
        super().__init__()
        layers = []
        last = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(last, h))
            layers.append(nn.ReLU())
            last = h
        layers.append(nn.Linear(last, output_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

# -------------------------
# 4) Prepare data (scaling)
# -------------------------
X = df[["theta","d","t"]].to_numpy()
y = df["E"].to_numpy().reshape(-1,1).astype(np.float32) / 100.0  # scale to 0..1

scaler_X = StandardScaler()
Xs = scaler_X.fit_transform(X)
cand_Xs = scaler_X.transform(candidates)

X_tensor = torch.tensor(Xs, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

# -------------------------
# 5) Training loop with Adam, scheduler and early stopping
# -------------------------
def train_model(model, X_tensor, y_tensor, lr=1e-3, weight_decay=1e-5,
                batch_size=32, epochs=2000, patience=200, verbose=True):
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=100)
    best_loss = float("inf")
    best_state = None
    wait = 0
    N = X_tensor.shape[0]

    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(N)
        epoch_loss = 0.0
        for i in range(0, N, batch_size):
            idx = perm[i:i+batch_size]
            xb = X_tensor[idx]
            yb = y_tensor[idx]
            preds = model(xb)
            loss = criterion(preds, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            epoch_loss += loss.item() * xb.shape[0]
        epoch_loss /= N
        scheduler.step(epoch_loss)

        if verbose and (epoch % 200 == 0 or epoch == epochs-1):
            print(f"Epoch {epoch:4d}  loss={epoch_loss:.6f}  lr={opt.param_groups[0]['lr']:.5g}")

        if epoch_loss < best_loss - 1e-8:
            best_loss = epoch_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait > patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch}, best loss {best_loss:.6f}")
                break
    model.load_state_dict(best_state)
    return model, best_loss

# -------------------------
# 6) Instantiate & train
# -------------------------
torch.manual_seed(0)
model = MLP(input_dim=3, hidden_sizes=[64,32], output_dim=1)
model, best_loss = train_model(model, X_tensor, y_tensor, lr=1e-3, weight_decay=1e-6,
                              batch_size=16, epochs=4000, patience=500, verbose=True)
print("Training finished. Best loss:", best_loss)

# -------------------------
# 7) Predict on candidates and rank them
# -------------------------
model.eval()
with torch.no_grad():
    cand_tensor = torch.tensor(cand_Xs, dtype=torch.float32)
    pred_cand = model(cand_tensor).numpy().reshape(-1) * 100.0

candidates_df["pred_E"] = pred_cand

from sklearn.neighbors import NearestNeighbors
nnbr = NearestNeighbors(n_neighbors=1).fit(X)
dists, _ = nnbr.kneighbors(candidates_df[["theta","d","t"]].to_numpy())
candidates_df["dist_to_train"] = dists.reshape(-1)

candidates_df_sorted = candidates_df.sort_values("pred_E", ascending=False).reset_index(drop=True)
print("\nTop candidate predictions (by NN):")
print(candidates_df_sorted.head(10))

top5 = candidates_df_sorted.head(5)
final3 = candidates_df_sorted.head(3)
print("\nFinal 3 selected (NN best guesses):")
print(final3[["theta","d","t","pred_E","dist_to_train"]])

# -------------------------
# 8) Plot: original points, candidates, and top picks
# -------------------------
fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=df["theta"], y=df["d"], z=df["t"],
    mode="markers",
    marker=dict(size=5, color=df["E"], colorscale="Viridis", colorbar=dict(title="E")),
    text=[f"E={e:.1f}" for e in df["E"]],
    name="Existing"
))
fig.add_trace(go.Scatter3d(
    x=candidates_df["theta"], y=candidates_df["d"], z=candidates_df["t"],
    mode="markers",
    marker=dict(size=3, color=candidates_df["pred_E"], colorscale="Cividis", opacity=0.7),
    text=[f"predE={pe:.1f}" for pe in candidates_df["pred_E"]],
    name="Candidates"
))
fig.add_trace(go.Scatter3d(
    x=top5["theta"], y=top5["d"], z=top5["t"],
    mode="markers+text",
    marker=dict(size=7, color="red", symbol="diamond"),
    text=[f"{i+1}" for i in range(len(top5))],
    textposition="top center",
    name="Top5"
))
fig.update_layout(scene=dict(xaxis_title="theta", yaxis_title="d", zaxis_title="t"),
                  title="Original points, Candidates, and Top NN picks")

out_html = "nn_candidates_plot.html"
fig.write_html(out_html, include_plotlyjs="cdn")
print(f"\nInteractive plot saved as: {out_html}  (open in browser)")

final3.to_csv("final3_candidates.csv", index=False)
print("Saved final3_candidates.csv")


Best 3 existing points (by measured E):
    theta     d     t     E
42   2.90  1.73  1.55  94.6
40   2.40  1.73  1.75  94.3
39   2.58  1.73  1.36  94.2

Sampled candidates (first 5):
   theta     d         t
0    2.9  1.73  1.350000
1    2.9  1.73  1.354040
2    2.9  1.73  1.358081
3    2.9  1.73  1.362121
4    2.9  1.73  1.366162
Epoch    0  loss=0.253685  lr=0.001
Epoch  200  loss=0.000342  lr=0.001
Epoch  400  loss=0.000114  lr=0.001
Epoch  600  loss=0.000074  lr=0.001
Epoch  800  loss=0.000054  lr=0.0005
Epoch 1000  loss=0.000041  lr=0.0005
Epoch 1200  loss=0.000040  lr=0.0005
Epoch 1400  loss=0.000033  lr=0.0005
Epoch 1600  loss=0.000042  lr=0.0005
Epoch 1800  loss=0.000014  lr=0.00025
Epoch 2000  loss=0.000014  lr=0.00025
Epoch 2200  loss=0.000012  lr=0.00025
Epoch 2400  loss=0.000006  lr=0.000125
Epoch 2600  loss=0.000005  lr=0.000125
Epoch 2800  loss=0.000003  lr=0.000125
Epoch 3000  loss=0.000003  lr=0.000125
Epoch 3200  loss=0.000002  lr=0.000125
Epoch 3400  loss=0.000002  lr

In [ ]:
# week 12
# 8) Plot: original points colored by E, sampled candidates (small), top picks highlighted
# -------------------------
import plotly.graph_objects as go

fig = go.Figure()

# 1️⃣ Original data
fig.add_trace(go.Scatter3d(
    x=df["theta"], y=df["d"], z=df["t"],
    mode="markers",
    marker=dict(size=5, color=df["E"], colorscale="Viridis", colorbar=dict(title="E")),
    hovertemplate=(
        "θ=%{x:.3f}<br>"
        "d=%{y:.3f}<br>"
        "t=%{z:.3f}<br>"
        "E=%{marker.color:.2f}<extra></extra>"
    ),
    name="Existing"
))

# 2️⃣ All candidates (small, semi-transparent)
fig.add_trace(go.Scatter3d(
    x=candidates_df["theta"], y=candidates_df["d"], z=candidates_df["t"],
    mode="markers",
    marker=dict(size=3, color=candidates_df["pred_E"], colorscale="Cividis", opacity=0.7),
    hovertemplate=(
        "θ=%{x:.3f}<br>"
        "d=%{y:.3f}<br>"
        "t=%{z:.3f}<br>"
        "predE=%{marker.color:.2f}<extra></extra>"
    ),
    name="Candidates"
))

# 3️⃣ Top5 (highlighted red diamonds)
fig.add_trace(go.Scatter3d(
    x=top5["theta"], y=top5["d"], z=top5["t"],
    mode="markers+text",
    marker=dict(size=8, color="red", symbol="diamond"),
    text=[f"{i+1}" for i in range(len(top5))],
    textposition="top center",
    hovertemplate=(
        "θ=%{x:.3f}<br>"
        "d=%{y:.3f}<br>"
        "t=%{z:.3f}<br>"
        "predE (Top)=%{text}<extra></extra>"
    ),
    name="Top5"
))

# 4️⃣ Layout
fig.update_layout(
    scene=dict(
        xaxis_title="θ",
        yaxis_title="d",
        zaxis_title="t"
    ),
    title="Original Points, Candidates, and Top NN Picks",
    legend=dict(x=0.02, y=0.98)
)

fig.show()  # HTML fayl o‘rniga to‘g‘ridan to‘g‘ri Colab’da chiqaradi


In [23]:
import numpy as np

def efficiency(x):
    # Convert MATLAB-style array definition and element-wise division to NumPy
    # x = (x - np.array([0, 1, 1])) / np.array([5, 2, 3])
    # Assuming x is already a numpy array, and the values [0;1;1] and [5;2;3] are constants
    # The values in [0;1;1] and [5;2;3] are from the problem context, indicating x is a 3-element vector
    x = (x - np.array([0, 1, 1])) / np.array([5, 2, 3])

    # Set up hyperparameters
    N = 10
    a = 0.3
    w = 1.5
    normconst = 1.14

    # Read coefficients from 'pdata.csv' using numpy (assuming it's a flat list of numbers)
    A = np.loadtxt('pdata.csv', delimiter=',')
    # Reshape the 1D array A into a 3D array (N+1 x N+1 x N+1)
    p = A.reshape(N + 1, N + 1, N + 1)

    # Compute raw f
    f = 0
    for k1 in range(N + 1):
        for k2 in range(N + 1):
            for k3 in range(N + 1):
                if k1 + k2 + k3 <= N:
                    # Convert MATLAB 1-based indexing p(k1+1,k2+1,k3+1) to Python 0-based p[k1,k2,k3]
                    # Convert MATLAB x(1)^k1 to Python x[0]**k1
                    f = f + p[k1, k2, k3] * (x[0]**k1) * (x[1]**k2) * (x[2]**k3)

    # Adjust f to be 0 < f <= 100
    # Convert MATLAB f^2 to Python f**2
    f = a / (a + f**2)
    # Convert MATLAB np.linalg.norm(x)^2 to Python np.linalg.norm(x)**2
    r = np.linalg.norm(x)**2 / (w**2)
    f = f * np.exp(-r)
    f = f * normconst
    f = f * 100
    return -f # as the nelder_mead function is minimization optimizer
if __name__ == "__main__":
    # print(efficiency(np.array([3.19, 2.50, 2.44])))
    # print(efficiency(np.array([2.11, 2.10, 2.93])))
    # Initial guess, choosing best point so far
    # x0 = np.array([2.90, 1.73, 1.55])  # we start from the best point so far
    x0 = np.array([1, 1, 1])  # we start from an arbitrary point
    n_dim = len(x0)

    print("Running Nelder-Mead on the sphere function (should converge to x=0).")
    res = nelder_mead(efficiency, x0, initial_step=0.2, max_iter=2000, tol_f=1e-15, tol_x=1e-15, verbose=True)
    print("Result summary:")
    print("  success:", res["success"])
    print("  message:", res["message"])
    print("  nit:", res["nit"])
    print("  x_best:", res["x_best"])
    print("  f_best:", res["f_best"])
    print("res['history']:", res['history'])


Running Nelder-Mead on the sphere function (should converge to x=0).
iter    0: f_best = -3.028402e+01, f_std = 1.19e+00, x_spread = 2.83e-01
iter    1: f_best = -3.512251e+01, f_std = 2.41e+00, x_spread = 2.83e-01
iter    2: f_best = -4.299986e+01, f_std = 5.30e+00, x_spread = 4.90e-01
iter    3: f_best = -5.821012e+01, f_std = 1.06e+01, x_spread = 6.32e-01
iter    4: f_best = -9.450957e+01, f_std = 2.28e+01, x_spread = 1.02e+00
iter   50: f_best = -9.637979e+01, f_std = 9.03e-06, x_spread = 4.31e-03
iter  100: f_best = -9.637980e+01, f_std = 2.32e-13, x_spread = 7.31e-07
iter  150: f_best = -9.637980e+01, f_std = 1.74e-14, x_spread = 7.90e-13
Result summary:
  success: True
  message: Converged (0.0 < 1e-15, 2.220446049250313e-16< 1e-15, tol_volume reached).
  nit: 163
  x_best: [2.36370775 1.73550259 1.33105797]
  f_best: -96.3797985682744
res['history']: []


In [24]:
# Project 5 — Task 1: Nelder–Mead implementation (Colab Python cell)
#Implementation and usage examples.
import numpy as np
import math
from typing import Callable, Tuple, Dict, Any

def nelder_mead(
    func: Callable[[np.ndarray], float],
    x0: np.ndarray,
    initial_step: float = 0.05,
    max_iter: int = 1000,
    tol_f: float = 1e-9,
    tol_x: float = 1e-9,
    tol_volume: float = 1e-15,
    alpha: float = 1.0,    # reflection
    gamma: float = 2.0,    # expansion
    rho: float = 0.5,      # contraction
    sigma: float = 0.5,    # shrink
    verbose: bool = True
) -> Dict[str, Any]:
    """
    Nelder-Mead optimizer.

    Parameters
    ----------
    func : callable
        Black-box objective taking a 1-D numpy array and returning a scalar.
    x0 : np.ndarray
        Initial guess vector (1-D, length n).
    initial_step : float
        Scale to create the initial simplex (relative to x0).
    max_iter : int
        Maximum number of iterations (simplex operations).
    tol_f : float
        Stopping tolerance on function values (std dev of simplex f-values).
    tol_x : float
        Stopping tolerance on simplex vertex spread (max distance).
    tol_volume : float
        Stopping tolerance on simplex volume L.
    alpha, gamma, rho, sigma : floats
        Standard NM coefficients (reflection, expansion, contraction, shrink).
    verbose : bool
        If True, prints progress occasionally.

    Returns
    -------
    dict
        result dictionary containing:
          - x_best : estimated minimizer
          - f_best : objective at x_best
          - nit : number of iterations performed
          - success : bool
          - message : status message
          - history : list of (x_simplex, f_values) occasionally (for debugging)
    """

    # Ensure x0 is numpy array
    x0 = np.asarray(x0, dtype=float)
    n = x0.size

    # Build initial simplex: n+1 vertices
    simplex = np.zeros((n + 1, n), dtype=float)
    simplex[0] = x0.copy()
    # If x0 is zero vector, use absolute step; otherwise proportionally scale
    for i in range(1, n + 1):
        e = np.zeros(n, dtype=float)
        e[i - 1] = 1.0
        # create vertex offset along axis i-1
        delta = initial_step * (abs(x0[i - 1]) if abs(x0[i - 1]) > 1e-8 else 1.0)
        simplex[i] = x0 + delta * e

    # Evaluate objective at simplex vertices
    f_vals = np.array([func(v) for v in simplex], dtype=float)

    # Keep history occasionally for diagnostics (not too large)
    history = []
    iter_count = 0
    success = False
    message = "Max iterations reached."

    # Main NM loop
    while iter_count < max_iter:
        # Sort simplex by function values (ascending)
        idx = np.argsort(f_vals)
        simplex = simplex[idx]
        f_vals = f_vals[idx]

        x_best = simplex[0].copy()
        f_best = f_vals[0]
        x_worst = simplex[-1].copy()
        f_worst = f_vals[-1]
        x_second_worst = simplex[-2].copy()
        f_second_worst = f_vals[-2]

        # Diagnostics / stopping criteria
        f_std = float(np.std(f_vals))
        diff_from_first = simplex - simplex[0]
        x_spread = float(np.max(np.linalg.norm(diff_from_first, axis=1)))
        # norm_volume_simplex = float(abs(np.linalg.det(diff_from_first[1:])/math.factorial(n))/x_spread) # computing the volume L

        if verbose and (iter_count % 50 == 0 or iter_count < 5):
            print(f"iter {iter_count:4d}: f_best = {f_best:.6e}, f_std = {f_std:.2e}, x_spread = {x_spread:.2e}")

        # if f_std < tol_f and x_spread < tol_x and norm_volume_simplex < tol_volume:
        if f_std < tol_f and x_spread < tol_x:
        # if volume_simplex < tol_volume: # alternatively check if volume L is shrinking
            success = True
            message = f"Converged ({f_std} < {tol_f}, {x_spread}< {tol_x}, tol_volume reached)."
            # message = f"Converged (tol_volume reached: {volume_simplex:.2e} < {tol_volume:.2e})."
            break

        # Compute centroid of all points except worst
        centroid = np.mean(simplex[:-1], axis=0)

        # Reflection
        x_reflect = centroid + alpha * (centroid - x_worst)
        f_reflect = func(x_reflect)

        if f_reflect < f_best:
            # Expansion
            x_expand = centroid + gamma * (x_reflect - centroid)
            f_expand = func(x_expand)
            if f_expand < f_reflect:
                # accept expansion
                simplex[-1] = x_expand
                f_vals[-1] = f_expand
            else:
                # accept reflection
                simplex[-1] = x_reflect
                f_vals[-1] = f_reflect
        elif f_reflect < f_second_worst:
            # accept reflection (better than second worst)
            simplex[-1] = x_reflect
            f_vals[-1] = f_reflect
        else:
            # Contraction
            if f_reflect < f_worst:
                # outside contraction
                x_contract = centroid + rho * (x_reflect - centroid)
            else:
                # inside contraction
                x_contract = centroid + rho * (x_worst - centroid)
            f_contract = func(x_contract)

            if f_contract < f_worst:
                simplex[-1] = x_contract
                f_vals[-1] = f_contract
            else:
                # Shrink: move all points toward best
                for i in range(1, n + 1):
                    simplex[i] = simplex[0] + sigma * (simplex[i] - simplex[0])
                    f_vals[i] = func(simplex[i])

        iter_count += 1

        # Save occasional snapshots to history
        if iter_count % 200 == 0:
            history.append((simplex.copy(), f_vals.copy()))

    # final sort
    idx = np.argsort(f_vals)
    simplex = simplex[idx]
    f_vals = f_vals[idx]
    x_best = simplex[0].copy()
    f_best = float(f_vals[0])

    return {
        "x_best": x_best,
        "f_best": f_best,
        "nit": iter_count,
        "success": success,
        "message": message,
        "history": history
    }

# ---------------------------------------------------------------------
# Demo: Test the Nelder–Mead implementation on a simple function (sphere)
# ---------------------------------------------------------------------
if __name__ == "__main__":
    # Example black-box: sphere function centered at 0: f(x)=||x||^2
    def sphere(x):
        # Ensure x is numpy array
        x = np.asarray(x, dtype=float)
        return float(np.dot(x, x))

    # Initial guess
    n_dim = 5
    x0 = np.ones(n_dim) * 0.5  # we start away from minimum at zero just to test

    print("Running Nelder-Mead on the sphere function (should converge to x=0).")
    res = nelder_mead(sphere, x0, initial_step=0.2, max_iter=2000, tol_f=1e-9, tol_x=1e-8, verbose=True)
    print("Result summary:")
    print("  success:", res["success"])
    print("  message:", res["message"])
    print("  nit:", res["nit"])
    print("  x_best:", res["x_best"])
    print("  f_best:", res["f_best"])



Running Nelder-Mead on the sphere function (should converge to x=0).
iter    0: f_best = 1.250000e+00, f_std = 4.10e-02, x_spread = 1.00e-01
iter    1: f_best = 1.250000e+00, f_std = 4.04e-02, x_spread = 1.28e-01
iter    2: f_best = 1.250000e+00, f_std = 3.96e-02, x_spread = 1.34e-01
iter    3: f_best = 1.250000e+00, f_std = 3.87e-02, x_spread = 1.40e-01
iter    4: f_best = 1.250000e+00, f_std = 3.79e-02, x_spread = 1.41e-01
iter   50: f_best = 5.970551e-04, f_std = 5.09e-04, x_spread = 5.24e-02
iter  100: f_best = 2.378545e-07, f_std = 3.24e-08, x_spread = 8.53e-04
iter  150: f_best = 4.163284e-12, f_std = 6.59e-12, x_spread = 5.14e-06
iter  200: f_best = 1.614673e-15, f_std = 1.28e-15, x_spread = 6.57e-08
Result summary:
  success: True
  message: Converged (1.5419763459695673e-17 < 1e-09, 8.945652308397825e-09< 1e-08, tol_volume reached).
  nit: 225
  x_best: [ 4.64224976e-10 -1.67781420e-09  1.50164167e-10  2.52861001e-09
  3.20162803e-10]
  f_best: 9.549487406868207e-18


In [ ]:
# Given data (theta, d, t, E):
data = np.array([
    [3.19, 2.50, 2.44, 16.0],
    [2.11, 2.10, 2.93, 32.0],
    [2.86, 1.61, 3.18, 11.7],
    [3.07, 1.56, 2.71, 23.3],
    [2.95, 2.02, 2.77, 14.8],
    [2.20, 2.13, 3.04, 24.5],
    [2.95, 2.01, 3.16, 8.4 ],
    [2.62, 2.50, 3.75, 53.9],
    [0.07, 2.58, 3.72, 1.0 ],
    [4.95, 1.35, 1.35, 50.7],
    [0.48, 2.06, 1.32, 50.2],
    [4.78, 2.60, 1.83, 3.6 ],
    [2.54, 1.00, 1.00, 57.0],
    [0.57, 1.21, 3.68, 49.3],
    [4.95, 1.00, 1.00, 7.8 ],
    [2.50, 3.00, 3.30, 1.0 ],
    [0.00, 1.00, 2.20, 27.0],
    [2.75, 1.15, 1.06, 70.4],
    [2.36, 1.10, 1.00, 56.5],
    [4.97, 1.11, 3.95, 0.9],
    [1.95, 1.63, 1.00, 71.2],
    [1.39, 1.97, 2.27, 0.0],
    [5.0,  1.0,  4.0,  0.8],
    [2.70, 2.40, 1.16, 50.7],
    [2.51, 1.89, 1.08, 93.3],
    [1.79, 2.02, 4.00, 35.7],
    [2.49, 2.04, 1.00, 87.6],
    [2.74, 1.67, 1.00, 92.3],
    [2.18, 1.32, 1.33, 68.4],
    [2.76, 1.81, 1.03, 92.2],
    [2.56, 1.91, 1.09, 92.3],
    [2.93, 1.75, 1.11, 91.7],
    [2.81, 1.62, 1.22, 93.4],
    [1.22, 1.04, 2.36, 66.3],
    [2.67, 1.75, 1.41, 93.6],
    [2.40, 1.73, 1.48, 93.8],
    [3.88, 2.14, 1.18, 38.1],
    [5.00, 1.00, 2.45, 45.1],
    [2.32, 1.86, 1.43, 93.8],
    [2.58, 1.73, 1.36, 94.2],
    [2.40, 1.73, 1.75, 94.3],
    [2.82, 1.61, 1.48, 93.9]
])